# Natural Language Processing — Complete Practical Guide

**A hands-on notebook for students**

This notebook walks through the full NLP toolkit, one concept at a time, with runnable code and a real project at the end.

### What you will learn

| # | Topic | What it does |
|---|-------|--------------|
| 1 | **Tokenization** | Split text into words / sentences |
| 2 | **Stemming** | Chop words to their root (fast, crude) |
| 3 | **Lemmatization** | Reduce words to dictionary form (smart) |
| 4 | **Stopword Removal** | Drop low-information words (the, is, a) |
| 5 | **Bag of Words (BoW)** | Turn text into word-count vectors |
| 6 | **TF-IDF** | Weight words by importance |
| 7 | **Word Embeddings** | Word2Vec & GloVe — meaning as vectors |
| 8 | **POS Tagging** | Label each word (noun, verb, ...) |
| 9 | **Named Entity Recognition** | Find people, places, orgs |
| 10 | **Dependency Parsing** | Grammatical structure of a sentence |
| 11 | **Sentiment Analysis** | Lexicon-based + ML-based |
| 12 | **PROJECT** | SMS Spam Detector (NLTK + scikit-learn) |

> **How to use:** Run the cells top to bottom. Read the markdown *before* each code block — it explains the *why*, the code shows the *how*.

---

## Setup — install & download resources

Run this once. It installs the libraries and downloads the small NLTK data packs we need.

- **NLTK** — classic teaching toolkit (tokenizers, stemmers, corpora)
- **scikit-learn** — vectorizers + ML models
- **gensim** — Word2Vec training
- **spaCy** — industrial NER & dependency parsing

In [ ]:
# Install (uncomment if running for the first time)
# !pip install nltk scikit-learn gensim spacy pandas matplotlib
# !python -m spacy download en_core_web_sm

import nltk

# Download the resources used throughout this notebook
for pkg in [
    'punkt', 'punkt_tab',            # tokenizers
    'stopwords',                     # stopword lists
    'wordnet', 'omw-1.4',            # lemmatizer dictionary
    'averaged_perceptron_tagger',    # POS tagger
    'averaged_perceptron_tagger_eng',
    'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words',  # NER chunker
    'vader_lexicon',                 # sentiment lexicon
]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f'Could not download {pkg}: {e}')

print('Setup complete.')

---
# 1. Tokenization

**Idea:** Computers can't read a sentence as one blob — we break it into pieces called **tokens** (usually words, sometimes sentences or sub-words). Tokenization is *always the first step* of any NLP pipeline.

**Why it matters:** Every later step (counting words, tagging, classifying) operates on tokens. Bad tokenization = bad everything downstream.

**Watch out for:** punctuation, contractions (`don't` → `do` + `n't`), and abbreviations (`U.S.A.`).

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

text = "Dr. Smith isn't happy. NLP is fun, and it's powerful! Visit openai.com today."

# Sentence tokenization — splits on sentence boundaries (note: it keeps 'Dr.' intact)
sentences = sent_tokenize(text)
print('SENTENCES:')
for i, s in enumerate(sentences, 1):
    print(f'  {i}. {s}')

# Word tokenization — splits into words + punctuation
words = word_tokenize(text)
print('\nWORDS:')
print(' ', words)

print(f'\n{len(sentences)} sentences, {len(words)} word-tokens')

---
# 2. Stemming

**Idea:** Reduce a word to its *root* by chopping off endings. `running`, `runs`, `ran`(no) → `run`. It uses crude rules, so the output is not always a real word (`studies` → `studi`).

**Trade-off:** Very fast, but sloppy. Good enough for search engines and rough text matching.

The **Porter stemmer** is the classic; **Snowball** is a slightly improved version.

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer('english')

words = ['running', 'runs', 'runner', 'easily', 'studies', 'studying', 'happily', 'connection']

print(f"{'WORD':<12}{'PORTER':<12}{'SNOWBALL':<12}")
print('-' * 36)
for w in words:
    print(f'{w:<12}{porter.stem(w):<12}{snowball.stem(w):<12}')

# Notice: 'studies' -> 'studi'  (not a real word — that's the crudeness of stemming)

---
# 3. Lemmatization

**Idea:** Like stemming, but *smart*. It uses a dictionary (WordNet) and the word's part-of-speech to return the real **base word** (the *lemma*). `studies` → `study`, `better` → `good`, `mice` → `mouse`.

**Trade-off:** Slower and needs a POS hint, but the output is always a valid word. Preferred when accuracy matters.

> **Key gotcha:** By default the lemmatizer assumes every word is a *noun*. Give it the correct POS tag (`v` for verb, `a` for adjective) and results improve dramatically.

In [ ]:
from nltk.stem import WordNetLemmatizer

lemma = WordNetLemmatizer()

print('WITHOUT POS (treated as noun):')
for w in ['running', 'studies', 'better', 'mice', 'was']:
    print(f'  {w:<10} -> {lemma.lemmatize(w)}')

print('\nWITH correct POS:')
print(f"  running (verb)  -> {lemma.lemmatize('running', pos='v')}")
print(f"  studies (verb)  -> {lemma.lemmatize('studies', pos='v')}")
print(f"  better  (adj)   -> {lemma.lemmatize('better', pos='a')}")
print(f"  was     (verb)  -> {lemma.lemmatize('was', pos='v')}")

print('\nStemming vs Lemmatization on "studies":')
print(f"  Porter stem   -> {porter.stem('studies')}   (not a real word)")
print(f"  Lemma (verb)  -> {lemma.lemmatize('studies', pos='v')}   (real word)")

---
# 4. Stopword Removal

**Idea:** Words like `the`, `is`, `at`, `a` appear everywhere and carry little meaning for tasks like classification. Removing them shrinks the data and focuses on content words.

**Caution:** Not always safe! For sentiment, `not` matters a lot (`not good` ≠ `good`). Remove stopwords only when they don't hurt your task.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print(f'NLTK has {len(stop_words)} English stopwords. A few:')
print(' ', sorted(list(stop_words))[:20])

sentence = "This is a simple example showing how we can remove the stopwords from a sentence"
tokens = word_tokenize(sentence.lower())

filtered = [w for w in tokens if w not in stop_words]

print(f'\nOriginal ({len(tokens)} tokens): {tokens}')
print(f'Filtered ({len(filtered)} tokens): {filtered}')

### Putting steps 1–4 together: a reusable `clean_text` function

This is the standard **preprocessing pipeline** you'll reuse in the project: lowercase → tokenize → remove non-alphabetic → remove stopwords → lemmatize.

In [ ]:
import re

def clean_text(text):
    """Full preprocessing pipeline: returns a clean, space-joined string."""
    text = text.lower()                              # 1. normalize case
    tokens = word_tokenize(text)                     # 2. tokenize
    tokens = [t for t in tokens if t.isalpha()]      # 3. keep letters only
    tokens = [t for t in tokens if t not in stop_words]  # 4. remove stopwords
    tokens = [lemma.lemmatize(t) for t in tokens]    # 5. lemmatize
    return ' '.join(tokens)

sample = "The cats were running quickly across 3 beautiful gardens!!!"
print('Raw:    ', sample)
print('Cleaned:', clean_text(sample))

---
# 5. Bag of Words (BoW)

**Idea:** ML models need numbers, not words. BoW builds a vocabulary of all words, then represents each document as a **count vector** — how many times each vocab word appears. Word order is thrown away (hence "bag").

**Example:** Vocabulary = `[dog, cat, runs]`. The sentence "dog runs" → `[1, 0, 1]`.

In scikit-learn this is `CountVectorizer`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

corpus = [
    'the cat sat on the mat',
    'the dog sat on the log',
    'cats and dogs are friends',
]

vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(corpus)

# Show as a readable table: rows = documents, columns = vocabulary words
df = pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out())
df.index = [f'doc{i+1}' for i in range(len(corpus))]
print('Vocabulary:', list(vectorizer.get_feature_names_out()))
print('\nBag-of-Words count matrix:')
df

---
# 6. TF-IDF (Term Frequency – Inverse Document Frequency)

**Problem with BoW:** Common words (like `the`) get high counts but tell us nothing distinctive.

**TF-IDF fixes this** by weighting each word:

$$\text{tfidf}(w, d) = \underbrace{tf(w, d)}_{\text{how often } w \text{ is in doc } d} \times \underbrace{\log\frac{N}{df(w)}}_{\text{rare across all docs} \Rightarrow \text{high weight}}$$

- A word frequent in **one** document but rare **overall** → **high** score (distinctive!)
- A word in **every** document (like `the`) → **low** score (useless).

TF-IDF almost always beats raw BoW for text classification.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
matrix = tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(matrix.toarray(),
                        columns=tfidf.get_feature_names_out()).round(2)
df_tfidf.index = [f'doc{i+1}' for i in range(len(corpus))]
print('TF-IDF matrix (higher = more distinctive to that document):')
df_tfidf

---
# 7. Word Embeddings — Word2Vec & GloVe

**The big limitation of BoW/TF-IDF:** they treat `king` and `queen` as totally unrelated (just different columns). They capture *counts*, not *meaning*.

**Word embeddings** map each word to a dense vector (e.g. 100 numbers) such that words with similar meaning have similar vectors. This is learned from *context* — "words that appear in similar surroundings have similar meaning."

The famous result:  `vector('king') - vector('man') + vector('woman') ≈ vector('queen')`

- **Word2Vec** (Google) — trains on your text using a neural net (CBOW / Skip-gram).
- **GloVe** (Stanford) — pre-trained on huge corpora; you just download and look up vectors.

Below we *train* a tiny Word2Vec on a toy corpus with `gensim`.

In [ ]:
from gensim.models import Word2Vec

# Toy corpus — each sentence is a list of tokens. (Real training needs millions of sentences.)
sentences = [
    ['king', 'is', 'a', 'strong', 'man'],
    ['queen', 'is', 'a', 'wise', 'woman'],
    ['boy', 'will', 'be', 'a', 'man'],
    ['girl', 'will', 'be', 'a', 'woman'],
    ['prince', 'is', 'a', 'young', 'king'],
    ['princess', 'is', 'a', 'young', 'queen'],
    ['man', 'is', 'strong'],
    ['woman', 'is', 'wise'],
    ['prince', 'is', 'a', 'boy', 'who', 'will', 'be', 'king'],
    ['princess', 'is', 'a', 'girl', 'who', 'will', 'be', 'queen'],
]

model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, workers=1, seed=42, epochs=200)

print("Vector for 'king' (first 10 of 50 dims):")
print(model.wv['king'][:10].round(3))

print("\nWords most similar to 'king':")
for word, score in model.wv.most_similar('king', topn=3):
    print(f'  {word:<10} {score:.3f}')

print("\nSimilarity(king, queen) =", round(model.wv.similarity('king', 'queen'), 3))
print("Similarity(king, girl)  =", round(model.wv.similarity('king', 'girl'), 3))

### Using pre-trained GloVe (optional)

Training your own is fine for learning, but real projects use **pre-trained** embeddings trained on billions of words. `gensim` can download GloVe with one line. (Skip if you have no internet — the download is ~66 MB.)

In [ ]:
# Uncomment to download & explore real GloVe vectors
# import gensim.downloader as api
# glove = api.load('glove-wiki-gigaword-50')   # 50-dim GloVe, ~66MB
#
# print(glove.most_similar('computer', topn=5))
# # The classic analogy: king - man + woman = ?
# print(glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=3))
print('GloVe demo is commented out — uncomment to run (needs internet).')

---
# 8. Part-of-Speech (POS) Tagging

**Idea:** Label each word with its grammatical role — noun (`NN`), verb (`VB`), adjective (`JJ`), etc. This tells us *what job* each word does in a sentence.

**Used for:** better lemmatization, extracting noun phrases, feeding parsers, and understanding structure.

In [ ]:
from nltk import pos_tag

sentence = "The quick brown fox jumps over the lazy dog"
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)

print('POS tags:')
for word, tag in tagged:
    print(f'  {word:<10} {tag}')

# Handy: decode any tag with nltk.help.upenn_tagset('NN')
print('\nCommon tags: NN=noun, NNP=proper noun, VB=verb, JJ=adjective, DT=determiner, IN=preposition')

---
# 9. Named Entity Recognition (NER)

**Idea:** Find and classify real-world entities in text — **people**, **organizations**, **locations**, **dates**, **money**, etc.

Example: *"Apple was founded by Steve Jobs in California."* → `Apple` (ORG), `Steve Jobs` (PERSON), `California` (GPE).

We show both **NLTK** (quick) and **spaCy** (more accurate, industry standard).

In [ ]:
from nltk import ne_chunk

sentence = "Barack Obama was born in Hawaii and worked at Google in California."
tokens = word_tokenize(sentence)
tags = pos_tag(tokens)
tree = ne_chunk(tags)

print('NLTK named entities:')
for subtree in tree:
    if hasattr(subtree, 'label'):   # it's a named entity chunk
        entity = ' '.join(word for word, tag in subtree.leaves())
        print(f'  {entity:<18} -> {subtree.label()}')

In [ ]:
# spaCy version — cleaner and more accurate
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    doc = nlp("Apple was founded by Steve Jobs in California in 1976 and is worth $2 trillion.")
    print('spaCy named entities:')
    for ent in doc.ents:
        print(f'  {ent.text:<15} -> {ent.label_:<8} ({spacy.explain(ent.label_)})')
except (ImportError, OSError):
    print('spaCy model not installed. Run:  python -m spacy download en_core_web_sm')

---
# 10. Dependency Parsing

**Idea:** Reveal the **grammatical structure** — which word depends on which. It answers *"who did what to whom?"* by linking each word to its head (e.g. the subject and object of a verb).

Example: in *"The cat chased the mouse"*, `cat` is the **subject** (nsubj) of `chased`, and `mouse` is the **object** (dobj).

spaCy does this best.

In [ ]:
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    doc = nlp("The hungry cat quickly chased the little mouse.")
    print(f"{'WORD':<10}{'DEP':<12}{'HEAD':<10}{'POS'}")
    print('-' * 40)
    for token in doc:
        print(f'{token.text:<10}{token.dep_:<12}{token.head.text:<10}{token.pos_}')
    print('\nnsubj = nominal subject, dobj = direct object, amod = adjective modifier')
    # To visualize graphically in Jupyter:
    # from spacy import displacy; displacy.render(doc, style='dep', jupyter=True)
except (ImportError, OSError):
    print('spaCy model not installed. Run:  python -m spacy download en_core_web_sm')

---
# 11. Sentiment Analysis

**Goal:** Decide whether text is **positive**, **negative**, or **neutral**. Two main approaches:

### (a) Lexicon-based (VADER)
Uses a dictionary of words pre-scored for sentiment (`good`=+, `terrible`=−). It also understands intensifiers (`very`), negation (`not good`), and punctuation/caps (`GREAT!!!`). **No training needed** — great for social media / short text.

### (b) ML-based
Train a classifier on labeled examples (we do this in the project). **Learns from data**, adapts to your domain, usually more accurate — but needs labeled training data.

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

reviews = [
    'This movie was absolutely fantastic! I loved it.',
    'Worst film ever. Total waste of time.',
    'The movie was okay, nothing special.',
    'It was not bad at all, actually pretty good!',
]

print('LEXICON-BASED (VADER) sentiment:\n')
for r in reviews:
    scores = sia.polarity_scores(r)
    compound = scores['compound']   # single summary score in [-1, +1]
    label = 'POSITIVE' if compound >= 0.05 else 'NEGATIVE' if compound <= -0.05 else 'NEUTRAL'
    print(f'  {label:<9} (compound={compound:+.2f})  "{r}"')

---
# 12. PROJECT — SMS Spam Detector

## 🎯 Now we combine everything into a real, useful application.

**Problem:** Given a text message, automatically decide if it is **spam** (unwanted promo/scam) or **ham** (a normal, legitimate message).

**Why this project?** It's the *hello world* of NLP classification, it's genuinely useful (email/SMS filters use exactly this), and it exercises the whole pipeline:

```
  Raw text
     │  clean_text()  ← tokenize, stopwords, lemmatize (steps 1–4)
     ▼
  Clean text
     │  TF-IDF        ← feature extraction (step 6)
     ▼
  Number vectors
     │  Naive Bayes   ← ML model (text classification)
     ▼
  spam / ham  +  accuracy report
```

**Algorithm — Multinomial Naive Bayes:** the classic, fast, surprisingly strong baseline for text. It learns which words are common in spam vs ham and applies Bayes' theorem.

> We build a small labeled dataset inline so the notebook runs anywhere. To use the real thing, download the [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) and load it with pandas (code shown at the bottom).

### Step 1 — Build the dataset

In [ ]:
import pandas as pd

# A small, illustrative dataset. label: 1 = spam, 0 = ham (legitimate)
data = [
    ('Congratulations! You won a FREE $1000 gift card. Click here to claim now!', 1),
    ('WINNER!! You have been selected for a cash prize. Call 090012345 now', 1),
    ('URGENT! Your account will be suspended. Verify your password immediately', 1),
    ('Get cheap loans with 0% interest! Reply YES to subscribe', 1),
    ('Free entry in a weekly competition to win an iPhone. Text WIN to 80085', 1),
    ('Claim your free ringtone now! Limited offer, click the link', 1),
    ('You have won a lottery of 5 million dollars. Send your bank details', 1),
    ('Hot singles in your area want to meet you! Sign up free today', 1),
    ('Discount 50% off on all products! Buy now before offer expires', 1),
    ('Your loan is approved! Click to receive money instantly no credit check', 1),
    ('Hey, are we still meeting for lunch tomorrow?', 0),
    ('Can you send me the notes from today\'s lecture?', 0),
    ('I will be a little late for the meeting, stuck in traffic', 0),
    ('Happy birthday! Hope you have a wonderful day', 0),
    ('Did you finish the assignment? I need some help with question 3', 0),
    ('Let\'s catch up this weekend, it has been a while', 0),
    ('Mom asked if you are coming home for dinner tonight', 0),
    ('The project deadline was moved to next Friday', 0),
    ('Thanks for your help earlier, really appreciate it', 0),
    ('Can we reschedule our call to 3pm instead of 2pm?', 0),
    ('See you at the gym after work today', 0),
    ('Please review the document and share your feedback', 0),
]

df = pd.DataFrame(data, columns=['message', 'label'])
print(f'Total messages: {len(df)}')
print(f"Spam: {df['label'].sum()}  |  Ham: {(df['label']==0).sum()}")
df.head()

### Step 2 — Clean every message (reuse our `clean_text` pipeline from steps 1–4)

In [ ]:
df['clean'] = df['message'].apply(clean_text)
df[['message', 'clean']].head()

### Step 3 — Split into training and test sets

We train the model on one part of the data and test it on *unseen* messages — that's how we honestly measure performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'],
    test_size=0.3, random_state=42, stratify=df['label']
)
print(f'Training messages: {len(X_train)}')
print(f'Test messages:     {len(X_test)}')

### Step 4 — Build a pipeline: TF-IDF → Naive Bayes

A scikit-learn `Pipeline` chains the vectorizer and the model so we fit and predict in one clean step (and it correctly learns the vocabulary from training data only).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

spam_clf = Pipeline([
    ('tfidf', TfidfVectorizer()),   # text  -> TF-IDF vectors
    ('nb',    MultinomialNB()),     # vectors -> spam / ham
])

spam_clf.fit(X_train, y_train)
print('Model trained!')

### Step 5 — Evaluate on the unseen test set

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = spam_clf.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.1%}\n')
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=['ham', 'spam'], zero_division=0))
print('Confusion matrix [rows=actual, cols=predicted]:')
print(pd.DataFrame(confusion_matrix(y_test, y_pred),
                   index=['actual ham', 'actual spam'],
                   columns=['pred ham', 'pred spam']))

### Step 6 — Try it on brand-new messages! 🎉

This is the payoff: feed the trained model messages it has never seen and watch it classify them.

In [ ]:
def predict_message(text):
    cleaned = clean_text(text)
    pred = spam_clf.predict([cleaned])[0]
    prob = spam_clf.predict_proba([cleaned])[0][pred]
    label = 'SPAM 🚫' if pred == 1 else 'HAM ✅'
    return label, prob

new_messages = [
    'Congratulations! You have won a free vacation. Click to claim your prize now!',
    'Hey, can you call me back when you get a chance?',
    'URGENT: verify your bank account or it will be locked',
    'Are you free for coffee tomorrow morning?',
    'Win a brand new car! Text CAR to 55555 now',
]

for msg in new_messages:
    label, prob = predict_message(msg)
    print(f'{label}  (confidence {prob:.0%})')
    print(f'   "{msg}"\n')

### (Optional) Use the real UCI dataset for serious accuracy

Our inline dataset is tiny (for demo). With the real 5,574-message dataset you'll see ~98% accuracy. Download it, then:

In [ ]:
# 1. Download 'SMSSpamCollection' from:
#    https://archive.ics.uci.edu/dataset/228/sms+spam+collection
# 2. Load it (tab-separated: label \t message):
#
# real_df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])
# real_df['label'] = real_df['label'].map({'ham': 0, 'spam': 1})
# real_df['clean'] = real_df['message'].apply(clean_text)
#
# Then re-run Steps 3–6 with real_df. Everything else stays the same!
print('See comments above to plug in the full UCI dataset.')

---
# Summary & Next Steps

### What you built
You went from raw text all the way to a working ML spam classifier, touching every core NLP concept:

**Preprocessing** (tokenize → stem/lemmatize → remove stopwords) → **Feature extraction** (BoW / TF-IDF / embeddings) → **Linguistic analysis** (POS, NER, parsing) → **Applications** (sentiment, classification).

### Key takeaways
- **Lemmatization > stemming** when you need real words.
- **TF-IDF > Bag of Words** for most classification tasks.
- **Embeddings (Word2Vec/GloVe)** capture *meaning*, unlike count-based methods.
- **Naive Bayes + TF-IDF** is a shockingly strong, fast baseline for text classification.

### Ideas to extend this project
1. Swap the full UCI dataset in and compare accuracy.
2. Try other models: `LogisticRegression`, `LinearSVC`, `RandomForestClassifier`.
3. Add `ngram_range=(1,2)` to `TfidfVectorizer` to capture two-word phrases.
4. Build **sentiment classification** on movie reviews using the exact same pipeline.
5. Replace TF-IDF with averaged Word2Vec vectors as features.
6. Wrap `predict_message()` in a tiny **Streamlit** web app.

Happy learning! 🚀